In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"] = "/media/ubuntu/BE6A02386A01EDC9/huggingface"
os.environ['HF_HUB_CACHE'] = "/media/ubuntu/BE6A02386A01EDC9/huggingface/hub"
os.environ['HF_DATASET_HOME'] = "/media/ubuntu/BE6A02386A01EDC9/huggingface/datasets"

import timeout_decorator
from timeout_decorator import TimeoutError

import sys
sys.path.append("../../../../LoRO")
sys.path.append('..')

from utils import *

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("Creekside/Qwen-3B-gsm8k-GRPO")
model = AutoModelForCausalLM.from_pretrained("Creekside/Qwen-3B-gsm8k-GRPO")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-0

In [5]:
dataset = load_dataset("GSM8K","main")['test']
print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})


In [6]:
question_answerer = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [7]:
idx = 679
question = dataset[idx]['question']

expected_answer =  dataset[idx]['answer']

prompt = [{ "role" : "user" , "content" : '{} Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.'}]
print(prompt)
print(expected_answer)
print(prompt[0]['content'].format(question))

[{'role': 'user', 'content': '{} Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.'}]
First find the number of hash browns per potato: 36 hash browns / 6 potatoes = <<36/6=6>>6 hash browns/potato
Then multiply that number by the number of potatoes to find the total number of hash browns: 6 hash browns/potato * 96 potatoes = <<6*96=576>>576 hash browns
#### 576
If 6 potatoes makes 36 hash browns, how many hash browns can you make out of 96 potatoes? Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.


In [8]:
query = prompt
query[0]['content'] = query[0]['content'].format(question)
answer = question_answerer(query, do_sample=False)
# print(answer[0]['generated_text'][1]['content'])
print(answer[0]['generated_text'][1]['content'].split('####')[-1])

576


In [9]:

timeout_seconds  = 60

@timeout_decorator.timeout(timeout_seconds, timeout_exception=TimeoutError)
def inference(pipe, query):
    return pipe(query, do_sample=False)

correct = 0
total = 0

model.eval()

progress_bar = tqdm.tqdm(range(len(dataset)))

for i in progress_bar:
    
    total += 1
    
    question = dataset[i]['question']
    expected_answer =  dataset[i]['answer'].split('#### ')[-1]
    
    query = [{ "role" : "user" , "content" : '{} Please think step by step, give the final number in a new line after #### without any other words.'.format(question)}]
    try:
        answer = inference(question_answerer, query)[0]['generated_text'][1]['content'].split('####')[-1].strip()
    except:
        print("TimeoutError index:{}".format(i))

    if answer == expected_answer:
        correct += 1
    
    progress_bar.set_postfix({'correct': correct, 'total': total, 'acc': correct/total})

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

 72%|███████▏  | 951/1319 [47:44<1:57:49, 19.21s/it, correct=674, total=951, acc=0.709]

TimeoutError index:950


100%|██████████| 1319/1319 [1:06:41<00:00,  3.03s/it, correct=936, total=1319, acc=0.71] 

correct:936, total:1319, accuracy:0.709628506444276


: 